# Proxipal Policy Optimization (PPO)

### 0 - Install Dependencies

- torch 2.8.0: https://pytorch.org/get-started/locally/
- Gymnasium: 
    - `pip install gymnasium[classic-control]`
    - `pip install swig`, `pip install gymnasium[box2d]`
- matplotlib: `pip install matplotlib`
 

In [ ]:
from abc import ABC, abstractmethod
import os

import numpy as np
import torch
import torch.nn as nn

import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics, RecordVideo

import matplotlib.pyplot as plt

### 1 - Getting familiar with the Gymnasium environments

Might need `pip install moviepy` to generate the videos.

In [ ]:
# env_name, render_mode  = "CartPole-v1", "rgb_array"
env_name, render_mode  = "BipedalWalker-v3", "rgb_array"

# initialize the environment, get state and action dimensions
env = gym.make(env_name, render_mode=render_mode)
state_dim = env.observation_space.shape[0]
# check if the action space is continuous or discrete
if isinstance(env.action_space, gym.spaces.Box):
    continuous_action = True
    action_dim = env.action_space.shape[0]
elif isinstance(env.action_space, gym.spaces.Discrete):
    continuous_action = False
    action_dim = env.action_space.n
else:
    raise NotImplementedError("Unknown action space type")

# Run several episodes and record videos using a random policy
num_eval_episodes = 5
video_folder = f"./videos/{env_name}/"
env = RecordVideo(
    env,
    video_folder=video_folder,     # Folder to save videos
    name_prefix="random_policy", # Prefix for video filenames
    episode_trigger=lambda x: True    # Record every episode
)
env = RecordEpisodeStatistics(env, buffer_length=num_eval_episodes)

print(f"Starting evaluation for {num_eval_episodes} episodes...")
print(f"Videos will be saved to: {video_folder}")

for episode_num in range(num_eval_episodes):
    env.reset()
    episode_reward = 0
    step_count = 0

    episode_done = False
    while not episode_done:
        action = env.action_space.sample()  # Random policy for demonstration
        _, reward, terminated, truncated, _ = env.step(action)
        episode_reward += reward
        step_count += 1
        episode_done = terminated or truncated
    print(f"Episode {episode_num + 1}: {step_count} steps, reward = {episode_reward}")
env.close()

# Print summary statistics
avg_reward = np.mean(env.return_queue)
avg_length = np.mean(env.length_queue)
std_reward = np.std(env.return_queue)
print(f'\nEvaluation Summary:')
print(f'Episode rewards: {list(env.return_queue)}')
print(f'Episode lengths: {list(env.length_queue)}')
print(f'\nAverage reward: {avg_reward:.2f} ± {std_reward:.2f}')
print(f'Average episode length: {avg_length:.1f} steps')

### 2 - Define actor-critic networks

In [ ]:
class ActorCritic(nn.Module):
    '''
        Actor-Critic Network for both discrete and continuous action space
        For continuous action space, we use diagonal Gaussian distribution with std fixed (not learnable)
        For discrete action space, we use Categorical distribution
    '''
    def __init__(self, state_dim, action_dim, continuous_action, action_std_init=1.0):
        super(ActorCritic, self).__init__()

        self.state_dim = state_dim      # Dimension of state 
        self.action_dim = action_dim    # For discrete action space, this is the number of actions, for continuous action space, this is the dimension of action
        self.continuous_action = continuous_action      # True: continuous action space; False: discrete action space
        self.set_action_std(action_std_init)   # Only takes effect for continuous action space
        
        # Actor:  a multi-layer perceptron (MLP) with two hidden layers
        # The dimension of the hidden layers is 64
        # The output layer is different for continuous and discrete action space
        if continuous_action:
            # self.actor outputs the mean of a diagonal Gaussian distribution
            # The std is fixed as self.action_std
            self.actor = nn.Sequential(
                            nn.Linear(state_dim, 64),
                            nn.Tanh(),
                            nn.Linear(64, 64),
                            nn.Tanh(),
                            nn.Linear(64, action_dim),
                            nn.Tanh()
                        )
        else:
            # self.actor outputs the probability of each action
            self.actor = nn.Sequential(
                            nn.Linear(state_dim, 64),
                            nn.Tanh(),
                            nn.Linear(64, 64),
                            nn.Tanh(),
                            nn.Linear(64, action_dim),
                            nn.Softmax(dim=-1)
                        )
            
        # Critic: a multi-layer perceptron (MLP) with two hidden layers
        # The dimension of the hidden layers is 64
        # The output layer is a single neuron that outputs the state value
        self.critic = nn.Sequential(
                        nn.Linear(state_dim, 64),
                        nn.Tanh(),
                        nn.Linear(64, 64),
                        nn.Tanh(),
                        nn.Linear(64, 1)
                    )
    
    def set_action_std(self, new_action_std):
        # Only takes effect for continuous action space
        if not self.continuous_action:
            return
        
        self.action_std = new_action_std
        # For MultivariateNormal, we need the covariance matrix, which is a diagonal matrix with action_std^2 on the diagonal
        self.action_var = torch.full((self.action_dim,), new_action_std * new_action_std) # (action_dim,)

    def actor_forward(self, state):
        '''
            Given a batch of states, forward it to the network and return a distribution object:            
            Args:
                - state: torch tensor of shape (batch_size, state_dim)
            Returns:
                - dist: a distribution object
                        torch.distributions.MultivariateNormal for continuous action space
                        torch.distributions.Categorical for discrete action space
        '''
        if self.continuous_action:
            # For continuous action space, we use diagonal Gaussian distribution
            action_mean = self.actor(state) # (batch_size, action_dim)
            cov_mat = torch.diag(
                self.action_var # (action_dim,) -> (action_dim, action_dim)
            ).unsqueeze(dim=0) # unsqueeze adds batch dimension, so cov_mat is (1, self.action_dim, self.action_dim)
            dist = torch.distributions.MultivariateNormal(action_mean, cov_mat)
        else:
            # For discrete action space, we use Categorical distribution
            action_probs = self.actor(state)
            dist = torch.distributions.Categorical(action_probs)
        
        return dist

    def act(self, state):
        '''
            Given a batch of states, forward it to the network and return a dict:            
            Args:
                - state: torch tensor of shape (batch_size, state_dim)
            Returns:
                - action: torch tensor of shape (batch_size, action_dim) for continuous action space
                          or 
                          torch tensor of shape (batch_size,) for discrete action space
                - action_logprob: torch tensor of shape (batch_size,) the log probability of the action
        '''
        dist = self.actor_forward(state)
        action = dist.sample()
        action_logprob = dist.log_prob(action)
        state_val = self.critic(state)

        return action, action_logprob, state_val

    def evaluate(self, state, action):
        '''
            Given a batch of states and actions, return the log probability of the actions, the state value, and the entropy of the action distribution
            This function is used during the update step, so needs to backpropagate through this function
            
            Args:
                - state:  torch tensor of shape (batch_size, state_dim)
                - action: torch tensor of shape (batch_size, action_dim) for continuous action space
                          or 
                          torch tensor of shape (batch_size,) for discrete action space
            Returns:
                - action_logprobs: torch tensor of shape (batch_size,) the log probability of the actions
                - state_values: torch tensor of shape (batch_size, 1) the state values
                - dist_entropy: torch tensor of shape (batch_size,) the entropy of the action distribution
        
        '''
        dist = self.actor_forward(state)
        action_logprobs = dist.log_prob(action)
        dist_entropy = dist.entropy()
        state_values = self.critic(state)
        
        return action_logprobs, state_values, dist_entropy
    

### 3 - Define rollout buffer

In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.states = []   # s_t
        self.actions = []  # a_t
        self.rewards = []  # r_t
        self.logprobs = [] # log pi(a_t|s_t)
        self.state_values = [] # V(s_t) estimated by the critic
        self.dones = []   # done mask indicates whether the episode is done after a_t  
    
    def clear(self):
        del self.actions[:]
        del self.states[:]
        del self.logprobs[:]
        del self.rewards[:]
        del self.state_values[:]
        del self.dones[:]

### 4 - Define general RL agent and PPO agent

In [ ]:
class Agent(ABC):
    @abstractmethod
    def select_action(self, state):
        '''
            Given a state, select an action
            Might update agent's internal variables (e.g., store state, action, logprob in buffer) 
        '''
        pass

    @abstractmethod
    def update(self):
        '''
            Update agent's internal variables after a number of transitions are collected
        '''
        pass

class PPOAgent(Agent):
    def __init__(self, state_dim, action_dim, continuous_action, 
                 action_std_init=0.6, lr_actor=0.0003, lr_critic=0.001, gamma=0.99, 
                 K_epochs=80, eps_clip=0.2, ent_coef=0.01):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.continuous_action = continuous_action
        
        self.action_std = action_std_init
        self.gamma = gamma
        self.eps_clip = eps_clip
        self.K_epochs = K_epochs
        self.ent_coef = ent_coef

        self.buffer = RolloutBuffer()

        self.policy = ActorCritic(state_dim, action_dim, continuous_action, action_std_init)
        self.optimizer = torch.optim.Adam([
                        {'params': self.policy.actor.parameters(), 'lr': lr_actor},
                        {'params': self.policy.critic.parameters(), 'lr': lr_critic}
                    ])
        
        self.policy_old = ActorCritic(state_dim, action_dim, continuous_action, action_std_init)
        self.policy_old.load_state_dict(self.policy.state_dict())

        self.MseLoss = nn.MSELoss()
    
    def select_action(self, state):
        '''
            Given a state, select an action using the old policy
            and store the state, action, log probability of the action in the buffer
            Args:
                - state: numpy array of shape (state_dim,)
            Returns:
                - action: numpy array of shape (action_dim,) for continuous action space
                          or 
                          numpy array of shape (1,) for discrete action space
        '''
        with torch.no_grad(): # No need to track gradients
            state = torch.from_numpy(state).unsqueeze(0) # (1, state_dim)
            action, action_logprob, state_value = self.policy_old.act(state)
            
            self.buffer.states.append(state)
            self.buffer.actions.append(action)
            self.buffer.logprobs.append(action_logprob)
            self.buffer.state_values.append(state_value)

            if self.continuous_action:
                return action.detach().cpu().numpy().squeeze(axis=0)  # (1, action_dim) -> (action_dim,)
            else:
                return action.item()

    def update(self):

        # Monte Carlo estimate of returns
        rewards = []
        discounted_reward = 0
        for reward, done in zip(reversed(self.buffer.rewards), reversed(self.buffer.dones)):
            if done:
                discounted_reward = 0
            discounted_reward = reward + (self.gamma * discounted_reward)
            rewards.insert(0, discounted_reward)
            
        # Normalizing the rewards
        rewards = torch.tensor(rewards, dtype=torch.float32)
        rewards = (rewards - rewards.mean()) / (rewards.std() + 1e-7)

        # convert list to tensor
        old_states = torch.squeeze(torch.stack(self.buffer.states, dim=0)).detach()
        old_actions = torch.squeeze(torch.stack(self.buffer.actions, dim=0)).detach()
        old_logprobs = torch.squeeze(torch.stack(self.buffer.logprobs, dim=0)).detach()
        old_state_values = torch.squeeze(torch.stack(self.buffer.state_values, dim=0)).detach()

        # calculate advantages
        advantages = rewards.detach() - old_state_values.detach()
        

        # Optimize policy for K epochs
        for _ in range(self.K_epochs):

            # Evaluating old actions and values
            logprobs, state_values, dist_entropy = self.policy.evaluate(old_states, old_actions)

            # match state_values tensor dimensions with rewards tensor
            state_values = torch.squeeze(state_values)
            
            # Finding the ratio (pi_theta / pi_theta__old)
            ratios = torch.exp(logprobs - old_logprobs.detach())

            # Finding Surrogate Loss   
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1-self.eps_clip, 1+self.eps_clip) * advantages

            # final loss of clipped objective PPO
            loss = -torch.min(surr1, surr2) + 0.5 * self.MseLoss(state_values, rewards) - 0.01 * dist_entropy
            
            # take gradient step
            self.optimizer.zero_grad()
            loss.mean().backward()
            self.optimizer.step()
            
        # Copy new weights into old policy
        self.policy_old.load_state_dict(self.policy.state_dict())

        # clear buffer
        self.buffer.clear()

    
    def set_action_std(self, new_action_std):
        if not self.continuous_action:
            return
        self.action_std = new_action_std
        self.policy.set_action_std(new_action_std)
        self.policy_old.set_action_std(new_action_std)
        
    def decay_action_std(self, action_std_decay_rate, min_action_std):
        if self.continuous_action:
            self.action_std = self.action_std - action_std_decay_rate
            if (self.action_std <= min_action_std):
                self.action_std = min_action_std
            self.set_action_std(self.action_std)
    
    
    def save(self, checkpoint_path):
        torch.save(self.policy_old.state_dict(), checkpoint_path)
   

    def load(self, checkpoint_path):
        self.policy_old.load_state_dict(torch.load(checkpoint_path, map_location=lambda storage, loc: storage))
        self.policy.load_state_dict(torch.load(checkpoint_path, map_location=lambda storage, loc: storage))
        

A unit test on PPO agent

In [ ]:
# A unit test on PPO agent
env_name, render_mode  = "CartPole-v1", "rgb_array"
env = gym.make(env_name, render_mode=render_mode)
state_dim = env.observation_space.shape[0]
# check if the action space is continuous or discrete
if isinstance(env.action_space, gym.spaces.Box):
    continuous_action = True
    action_dim = env.action_space.shape[0]
elif isinstance(env.action_space, gym.spaces.Discrete):
    continuous_action = False
    action_dim = env.action_space.n
else:
    raise NotImplementedError("Unknown action space type")

agent = PPOAgent(state_dim, action_dim, continuous_action)
state, _ = env.reset()
max_ep_len = 100
t = 0
for _ in range (max_ep_len):
    action = agent.select_action(state)
    next_state, reward, terminated, truncated, _ = env.step(action)
    state = next_state
    t += 1
    if terminated or truncated:
        break
env.close()
print("Total number of transitions stored in buffer:", len(agent.buffer.states))
print("PPO agent unit test passed!")

### 5 - Traing the PPO agent

In [ ]:
def train_ppo(args):
    # Get hyperparameters from args, or use default values
    env_name = args['env_name']
    max_ep_len = args.get('max_ep_len', 1000) # max timesteps in one episode
    max_training_timesteps = args.get('max_training_timesteps', int(3e6))   # break training loop if timeteps > max_training_timesteps
    log_freq = args.get('log_freq', max_ep_len * 2)       # log avg reward in the interval (in num timesteps)
    save_model_freq = args.get('save_model_freq', int(1e5))      # save model frequency (in num timesteps)
    action_std_init = args.get('action_std_init', 0.6)          # starting std for action distribution (Multivariate Normal) for continuous action space
    action_std_decay_rate = args.get('action_std_decay_rate', 0.05) # linearly decay action_std (action_std = action_std - action_std_decay_rate)
    min_action_std = args.get('min_action_std', 0.1)          # minimum action_std (stop decay after action_std <= min_action_std)
    action_std_decay_freq = args.get('action_std_decay_freq', int(2.5e5))   #  decay action_std every action_std_decay_freq timesteps
    update_freq = args.get('update_freq', max_ep_len * 4)   # update policy every n timesteps
    K_epochs = args.get('K_epochs', 80)               # update policy for K epochs
    eps_clip = args.get('eps_clip', 0.2)              # clip parameter for PPO
    gamma = args.get('gamma', 0.99)                # discount factor
    lr_actor = args.get('lr_actor', 0.0003)       # learning rate for actor network
    lr_critic = args.get('lr_critic', 0.001)       # learning rate for critic network

    ####### Initialization: Logging ######
    log_dir = f"./logs_and_checkpoints/PPO_{env_name}"
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
    # get number of runs in the directory, so we can create new log file for each run
    run_num = 0
    current_num_folders = next(os.walk(log_dir))[1]
    run_num = len(current_num_folders)
    log_dir += '/run{}'.format(run_num)
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
    log_f_name = log_dir + "/log_run" + str(run_num) + ".csv"
    log_f = open(log_f_name,"w+")
    print("logging at : " + log_f_name)

    ############## Initialization: Environment ############
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    if isinstance(env.action_space, gym.spaces.Box):
        continuous_action = True
        action_dim = env.action_space.shape[0]
    elif isinstance(env.action_space, gym.spaces.Discrete):
        continuous_action = False
        action_dim = env.action_space.n
    else:
        raise NotImplementedError("Unknown action space type!")

    ############## Initialization: PPO Agent ############
    agent = PPOAgent(state_dim, action_dim, continuous_action, action_std_init=action_std_init,
                    lr_actor = lr_actor, lr_critic=lr_critic, gamma=gamma, K_epochs=K_epochs, eps_clip=eps_clip)

    ############## Training loop ##################
    training_timestep = 0
    i_episode = 0
    log_running_reward = 0
    log_running_episodes = 0
    log_running_reward_list = [] # to plot later
    while training_timestep <= max_training_timesteps:
        # start a new episode
        state, _ = env.reset()
        ep_reward = 0.
        for _ in range (max_ep_len):
            # select action
            action = agent.select_action(state) # saves state, action, logprob, state_val in buffer
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # save reward and is_dones to buffer
            agent.buffer.rewards.append(reward)
            agent.buffer.dones.append(done)
            
            # transition to next state
            ep_reward += reward
            state = next_state
            training_timestep += 1

            # update PPO agent
            if training_timestep % update_freq == 0:
                agent.update() # this also clears the buffer

            # if continuous action space; then decay action std of ouput action distribution
            if continuous_action and training_timestep % action_std_decay_freq == 0:
                agent.decay_action_std(action_std_decay_rate, min_action_std)

            # log in logging file
            if training_timestep % log_freq == 0:
                # log average reward till last episode
                log_avg_reward = log_running_reward / log_running_episodes
                print('training_timestep {} , avg_ep_reward {:.4f}\n'.format(training_timestep, log_avg_reward))
                log_running_reward_list.append(log_avg_reward)
                log_f.write('{},{},{}\n'.format(i_episode, training_timestep, log_avg_reward))
                log_f.flush()

                log_running_reward = 0
                log_running_episodes = 0

        
            # save model weights
            if training_timestep % save_model_freq == 0:
                checkpoint_path = log_dir + "/{}.pth".format(training_timestep)
                print("--------------------------------------------------------------------------------------------")
                print("saving model at : " + checkpoint_path)
                agent.save(checkpoint_path)
                print("model saved")
                print("--------------------------------------------------------------------------------------------")
                
            # break if the episode is over
            if done:
                break

        log_running_reward += ep_reward
        log_running_episodes += 1

        i_episode += 1


    log_f.close()
    env.close()

    # plot log_running_reward_list
    y = np.array(log_running_reward_list)
    x = (np.arange(len(y))+1) * log_freq
    plt.plot(x,y)
    plt.xlabel('training timesteps')
    plt.ylabel('log_running_reward')
    plt.title('{}, run{}'.format(env_name, run_num))
    # save fig
    fig_path = log_dir + "/log_running_reward_run{}.png".format(run_num)
    plt.savefig(fig_path)
    print("figure saved at : " + fig_path)
    
    plt.show()



Training CartPole-v1 (took <1m on my Mac mini)

In [ ]:

###################################  Hyperparmeters for  CartPole-v1 ##########################################
args = {}
args['env_name'] = "CartPole-v1"
args['max_ep_len'] = 400    # max timesteps in one episode
args['max_training_timesteps'] = int(1e4)   # break training loop if timeteps > max_training_timesteps
args['log_freq'] = args['max_ep_len'] * 10       # log avg reward in the interval (in num timesteps)
args['save_model_freq'] = int(2e4)      # save model frequency (in num timesteps)
args['action_std_init'] = None
args['update_freq'] = args['max_ep_len'] * 4   # update policy every n timesteps
args['K_epochs'] = 40               # update policy for K epochs
args['eps_clip'] = 0.2              # clip parameter for PPO
args['gamma'] = 0.99                # discount factor
args['lr_actor'] = 0.0003       # learning rate for actor network
args['lr_critic'] = 0.001       # learning rate for critic network
###################################  END Hyperparmeters for  CartPole-v1 ##########################################

# train ppo agent on CartPole-v1
train_ppo(args)


Training CartPole-v1 (took <10m on my Mac mini)

In [ ]:

###################################  Hyperparmeters for  BipedalWalker-v3 ##########################################
args = {}
args['env_name'] = "BipedalWalker-v3"
args['max_ep_len'] = 1000    # max timesteps in one episode
args['max_training_timesteps'] = int(1e6)   # break training loop if timeteps > max_training_timesteps
args['log_freq'] = args['max_ep_len'] * 50       # log avg reward in the interval (in num timesteps)
args['save_model_freq'] = int(1e5)      # save model frequency (in num timesteps)
args['action_std_init'] = 0.6           # starting std for action distribution (Multivariate Normal) for continuous action space
args['action_std_decay_rate'] = 0.05    # linearly decay action_std (action_std = action_std - action_std_decay_rate)
args['min_action_std'] = 0.1            # minimum action_std (stop decay after action_std <= min_action_std)
args['action_std_decay_freq'] = int(2.5e5)   #  decay action_std every action_std_decay_freq timesteps
args['update_freq'] = args['max_ep_len'] * 4   # update policy every n timesteps
args['K_epochs'] = 80               # update policy for K epochs
args['eps_clip'] = 0.2              # clip parameter for PPO
args['gamma'] = 0.99                # discount factor
args['lr_actor'] = 0.0003       # learning rate for actor network
args['lr_critic'] = 0.001       # learning rate for critic network
###################################  END Hyperparmeters for BipedalWalker-v3 ##########################################

# train ppo agent on BipedalWalker-v3
train_ppo(args)